# 01. RunPod setup e dados

Notebook canonico para preparar o ambiente RunPod/Jupyter e executar a parte de dados do pipeline usando os CLIs oficiais do repositorio via `.venv/bin/python`.

In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    markers = ('pyproject.toml', 'README.md', 'scripts')
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    workspace = Path('/workspace')
    if workspace.exists():
        candidates.append(workspace.resolve())
        for child in sorted(workspace.iterdir()):
            if child.is_dir():
                candidates.append(child.resolve())
    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise FileNotFoundError('Nao foi possivel localizar a raiz do projeto a partir do notebook.')


PROJECT_ROOT = find_project_root()
VENV_PYTHON = PROJECT_ROOT / '.venv' / 'bin' / 'python'


def resolve_path(path: str | Path) -> Path:
    candidate = Path(path)
    return candidate if candidate.is_absolute() else PROJECT_ROOT / candidate


def assert_path(path: str | Path, *, kind: str | None = None) -> Path:
    resolved = resolve_path(path)
    if not resolved.exists():
        raise FileNotFoundError(f'Path not found: {resolved}')
    if kind == 'file' and not resolved.is_file():
        raise FileNotFoundError(f'Expected file, found: {resolved}')
    if kind == 'dir' and not resolved.is_dir():
        raise FileNotFoundError(f'Expected directory, found: {resolved}')
    print(f'OK: {resolved}')
    return resolved


def run_cmd(args: list[object], env: dict[str, object] | None = None, check: bool = True) -> subprocess.CompletedProcess:
    cmd = [str(arg) for arg in args]
    merged_env = os.environ.copy()
    if env:
        merged_env.update({key: str(value) for key, value in env.items() if value is not None})
    print('+', shlex.join(cmd))
    return subprocess.run(cmd, cwd=PROJECT_ROOT, env=merged_env, check=check, text=True)


def run_project_python(args: list[object], env: dict[str, object] | None = None) -> subprocess.CompletedProcess:
    assert_path(VENV_PYTHON, kind='file')
    return run_cmd([VENV_PYTHON, *args], env=env)


def preview_csv(path: str | Path, rows: int = 5, columns: list[str] | None = None) -> pd.DataFrame:
    resolved = assert_path(path, kind='file')
    frame = pd.read_csv(resolved)
    if columns:
        frame = frame.loc[:, columns]
    print(f'rows={len(frame)} columns={list(frame.columns)}')
    display(frame.head(rows))
    return frame


def preview_json(path: str | Path) -> dict:
    resolved = assert_path(path, kind='file')
    payload = json.loads(resolved.read_text(encoding='utf-8'))
    print(json.dumps(payload, indent=2, ensure_ascii=True)[:4000])
    return payload


def show_tree(path: str | Path, max_depth: int = 2, max_entries: int = 40) -> None:
    root = assert_path(path)
    print(root)
    base_depth = len(root.parts)
    shown = 0
    for child in sorted(root.rglob('*')):
        depth = len(child.parts) - base_depth
        if depth > max_depth:
            continue
        rel = child.relative_to(root)
        suffix = '/' if child.is_dir() else ''
        print(f"{'  ' * depth}{rel}{suffix}")
        shown += 1
        if shown >= max_entries:
            print(f'... truncated after {max_entries} entries')
            break


CONFIG_PATH = Path('configs/speecht5_minimal.yaml')
PROMPTS_PATH = Path('data/prompts/ptbr_test_prompts.csv')
RAW_DIR = Path('data/raw/common_voice_pt')
RAW_TSV = RAW_DIR / 'validated.tsv'
RAW_CLIPS_DIR = RAW_DIR / 'clips'
RAW_METADATA_PATH = Path('data/manifests/common_voice_metadata.csv')
PROCESSED_DIR = Path('data/processed/common_voice_pt')
PROCESSED_METADATA_PATH = Path('data/manifests/common_voice_processed.csv')
MANIFEST_PATH = Path('data/manifests/data_manifest.csv')
SPEAKER_SELECTION_PATH = Path('data/manifests/speaker_selection.csv')
INVENTORY_JSON_PATH = Path('artifacts/dataset_inventory.json')
EMBEDDINGS_DIR = Path('artifacts/embeddings')
EMBEDDINGS_INDEX_PATH = EMBEDDINGS_DIR / 'speaker_embeddings.csv'
RUN_MATRIX_PATH = Path('artifacts/run_matrix.csv')
SAMPLES_PATH = Path('artifacts/evaluation/samples.csv')
SPEAKER_TARGET_COUNT = 1000

print(f'PROJECT_ROOT={PROJECT_ROOT}')
print(f'VENV_PYTHON={VENV_PYTHON}')


## Contexto RunPod

A celula abaixo confirma a raiz detectada do projeto e mostra os caminhos esperados. O notebook funciona mesmo quando o diretorio corrente do Jupyter nao e exatamente a raiz do repo.

In [ ]:
workspace_root = Path('/workspace').resolve()
print(f'cwd={Path.cwd().resolve()}')
print(f'project_root={PROJECT_ROOT}')
print(f'under_workspace={PROJECT_ROOT == workspace_root or workspace_root in PROJECT_ROOT.parents}')
for relative_path in [CONFIG_PATH, PROMPTS_PATH, Path('scripts'), Path('tcc_audio')]:
    print(f'{relative_path} -> {resolve_path(relative_path)}')


## Bootstrap opcional

Se a `.venv` do projeto ainda nao existir no pod, a proxima celula executa `bash scripts/bootstrap_runpod.sh`. Se a `.venv` ja existir, ela apenas valida o interpretador e mostra a versao do Python. O kernel do notebook continua sendo o Python do Jupyter host.

In [ ]:
if VENV_PYTHON.exists():
    print('`.venv` ja esta pronta; pulando bootstrap do host RunPod.')
else:
    run_cmd(['bash', 'scripts/bootstrap_runpod.sh'])

assert_path(VENV_PYTHON, kind='file')
run_project_python(['-V'])


## Token Mozilla Data Collective

Preencha `MOZILLA_DATA_COLLECTIVE_API_KEY` abaixo caso o token ainda nao esteja definido no ambiente do pod. O valor fica disponivel apenas para este processo do notebook.

In [ ]:
MOZILLA_DATA_COLLECTIVE_API_KEY = os.environ.get('MOZILLA_DATA_COLLECTIVE_API_KEY', '')

if MOZILLA_DATA_COLLECTIVE_API_KEY:
    os.environ['MOZILLA_DATA_COLLECTIVE_API_KEY'] = MOZILLA_DATA_COLLECTIVE_API_KEY
    print('Token Mozilla Data Collective carregado no ambiente do notebook.')
else:
    print('Preencha MOZILLA_DATA_COLLECTIVE_API_KEY nesta celula antes do download.')


## 1. Download do Common Voice PT

In [ ]:
run_project_python([
    'scripts/download_common_voice_pt.py',
    '--out-dir', RAW_DIR,
], env={'MOZILLA_DATA_COLLECTIVE_API_KEY': os.environ.get('MOZILLA_DATA_COLLECTIVE_API_KEY', '')})

assert_path(RAW_DIR, kind='dir')
assert_path(RAW_TSV, kind='file')
assert_path(RAW_CLIPS_DIR, kind='dir')
show_tree(RAW_DIR, max_depth=2, max_entries=25)


## 2. Preparar metadados do subconjunto pt-BR

In [ ]:
run_project_python([
    'scripts/prepare_common_voice_metadata.py',
    '--tsv', RAW_TSV,
    '--clips-dir', RAW_CLIPS_DIR,
    '--locale', 'pt',
    '--variant', 'pt-BR',
    '--out', RAW_METADATA_PATH,
])

preview_csv(RAW_METADATA_PATH, columns=['source_speaker_id', 'utterance_id', 'duration_s', 'audio_path', 'variant'])


## 3. Preprocessar audio para WAV mono

In [ ]:
run_project_python([
    'scripts/preprocess_audio_dataset.py',
    '--metadata', RAW_METADATA_PATH,
    '--out-dir', PROCESSED_DIR,
    '--out-metadata', PROCESSED_METADATA_PATH,
])

assert_path(PROCESSED_DIR, kind='dir')
preview_csv(PROCESSED_METADATA_PATH, columns=['source_speaker_id', 'utterance_id', 'duration_s', 'audio_path'])


## 4. Selecionar speakers e materializar manifesto

In [ ]:
run_project_python([
    'scripts/select_speakers.py',
    '--metadata', PROCESSED_METADATA_PATH,
    '--speaker-target-count', SPEAKER_TARGET_COUNT,
    '--manifest-out', MANIFEST_PATH,
    '--speaker-selection-out', SPEAKER_SELECTION_PATH,
])

selection = preview_csv(SPEAKER_SELECTION_PATH, columns=['speaker_id', 'source_speaker_id', 'gender', 'reference_audio'])
manifest = preview_csv(MANIFEST_PATH, columns=['speaker_id', 'utterance_id', 'split', 'duration_s', 'audio_path'])
print(f'selected_speakers={selection["speaker_id"].nunique()} manifest_rows={len(manifest)}')


## 5. Registrar inventario hierarquico dos dados

In [ ]:
run_project_python([
    'scripts/log_dataset_inventory.py',
    '--config', CONFIG_PATH,
    '--json-out', INVENTORY_JSON_PATH,
])

preview_json(INVENTORY_JSON_PATH)


## 6. Validar manifesto e prompts

In [ ]:
run_project_python([
    'scripts/validate_manifest.py',
    '--manifest', MANIFEST_PATH,
    '--config', CONFIG_PATH,
    '--prompts', PROMPTS_PATH,
    '--check-files',
])

preview_csv(MANIFEST_PATH, columns=['speaker_id', 'utterance_id', 'split', 'text_variant'])


## 7. Extrair embeddings de speaker

In [ ]:
run_project_python([
    'scripts/extract_speaker_embeddings.py',
    '--config', CONFIG_PATH,
    '--speaker-selection', SPEAKER_SELECTION_PATH,
    '--out-index', EMBEDDINGS_INDEX_PATH,
    '--out-dir', EMBEDDINGS_DIR,
])

assert_path(EMBEDDINGS_DIR, kind='dir')
preview_csv(EMBEDDINGS_INDEX_PATH)


## 8. Gerar run matrix

In [ ]:
run_project_python([
    'scripts/generate_run_matrix.py',
    '--config', CONFIG_PATH,
    '--out', RUN_MATRIX_PATH,
])

run_matrix = preview_csv(RUN_MATRIX_PATH)
print(f'run_matrix_rows={len(run_matrix)} conditions={run_matrix["condition"].nunique()}')


## 9. Inicializar samples.csv

In [ ]:
run_project_python([
    'scripts/init_samples.py',
    '--run-matrix', RUN_MATRIX_PATH,
    '--speaker-selection', SPEAKER_SELECTION_PATH,
    '--speaker-embeddings', EMBEDDINGS_INDEX_PATH,
    '--out', SAMPLES_PATH,
])

samples = preview_csv(SAMPLES_PATH, columns=['sample_id', 'condition', 'speaker_id', 'prompt_id', 'status'])
print(samples['status'].value_counts(dropna=False))
